# Cortex AI Gateway + LangChain Agent + MCP

This notebook demonstrates how to build a **LangChain agent** that:

1. Routes inference through **Snowflake's Cortex AI Gateway** (using a Snowflake-hosted Claude model)
2. Connects to **Snowflake data via MCP** (Model Context Protocol) for tool use
3. Produces **observability traces** you can query in `SNOWFLAKE.TELEMETRY.AGENT_TRACE_TABLE`

## Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                     LangChain Agent (ReAct)                     │
│                                                                 │
│   LLM: ChatOpenAI ──► Cortex AI Gateway ──► Claude (Cortex)    │
│                                                                 │
│   Tools: MCP Client ──► Snowflake MCP Server                   │
│          ├── Cortex Analyst (SQL via semantic view)             │
│          └── Cortex Search (RAG over strategy docs)            │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
            SNOWFLAKE.TELEMETRY.AGENT_TRACE_TABLE
            SNOWFLAKE.ACCOUNT_USAGE.AI_GATEWAY_USAGE_HISTORY
```

## Prerequisites

1. Run `setup.sql` in your Snowflake account to create the database, tables, semantic view, search service, MCP server, and gateway route.
2. Create a [Personal Access Token (PAT)](https://docs.snowflake.com/en/user-guide/admin-pat) in Snowflake for API authentication.
3. Have your Snowflake account identifier (e.g., `myorg-myaccount`).

## 1. Install Dependencies

In [1]:
%pip install langchain-openai langchain-mcp-adapters langgraph snowflake-connector-python

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

We read credentials from `~/.snowflake/connections.toml` (the standard Snowflake CLI config).
Set `SNOWFLAKE_CONNECTION_NAME` to match the connection you want to use.

The AI Gateway has two distinct URL paths:
- **Inference:** `/api/v2/aigateways/snowflake/v1/chat/completions` (OpenAI-compatible) or `/v1/messages` (Anthropic)
- **Admin:** `/api/v2/aigateways/SNOWFLAKE` (spec management, SHOW, ALTER)

In [2]:
import os
import tomllib
from pathlib import Path

CONNECTION_NAME = os.environ.get("SNOWFLAKE_CONNECTION_NAME", "parker_demo")

# Read connection config from connections.toml
connections_path = Path.home() / ".snowflake" / "connections.toml"
if not connections_path.exists():
    # Fall back to Cortex agent config location
    connections_path = Path.home() / ".snowflake" / "cortex" / "agent" / "connections.toml"

with open(connections_path, "rb") as f:
    connections = tomllib.load(f)

conn_cfg = connections[CONNECTION_NAME]
SNOWFLAKE_ACCOUNT = conn_cfg["account"]
SNOWFLAKE_USER = conn_cfg["user"]
SNOWFLAKE_PAT = conn_cfg["password"]  # PAT token stored as password

# Two distinct gateway URLs (hyphens, not underscores, for valid SSL certs)
SF_HOST = f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com".replace('_', '-')
INFERENCE_BASE_URL = f"https://{SF_HOST}/api/v2/aigateways/snowflake/v1"
GATEWAY_ADMIN_URL = f"https://{SF_HOST}/api/v2/aigateways/SNOWFLAKE"

MODEL = "openai-gpt-5.4"  # Model name passed directly — the gateway allows all models via '*'

print(f"Connection:    {CONNECTION_NAME}")
print(f"Account:       {SNOWFLAKE_ACCOUNT}")
print(f"User:          {SNOWFLAKE_USER}")
print(f"Inference URL: {INFERENCE_BASE_URL}")
print(f"Admin URL:     {GATEWAY_ADMIN_URL}")
print(f"Model:         {MODEL}")

Connection:    parker_demo
Account:       SFSENORTHAMERICA-PERICKSON_AWS1
User:          PERICKSON
Inference URL: https://SFSENORTHAMERICA-PERICKSON-AWS1.snowflakecomputing.com/api/v2/aigateways/snowflake/v1
Admin URL:     https://SFSENORTHAMERICA-PERICKSON-AWS1.snowflakecomputing.com/api/v2/aigateways/SNOWFLAKE
Model:         openai-gpt-5.4


## 3. Initialize the Gateway LLM

The Cortex AI Gateway exposes an **OpenAI-compatible** API. We use `ChatOpenAI` from LangChain,
pointing `base_url` at the inference endpoint and passing the PAT as the API key.

The `model` parameter is the actual model name (e.g., `claude-sonnet-4-6`). The gateway's
`models` allowlist controls which models are permitted — our spec uses `'*'` to allow all.

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=MODEL,
    base_url=INFERENCE_BASE_URL,
    api_key=SNOWFLAKE_PAT,
    temperature=0,
    max_tokens=4096,
)

# Quick test — this goes through the gateway to Claude
response = llm.invoke("What is ROAS in marketing? Answer in one sentence.")
print(response.content)

ROAS (Return on Ad Spend) is a marketing metric that measures how much revenue is generated for every dollar spent on advertising.


## 4. Connect MCP Tools

We use `langchain-mcp-adapters` to connect to the **Snowflake MCP server** object we created
in `setup.sql` (`MARKETING_MCP`). This MCP server exposes two tools:

- **`query_campaigns`** (Cortex Analyst) — translates natural language to SQL against the semantic view
- **`search_strategy_docs`** (Cortex Search) — retrieves relevant strategy documents via RAG

The MCP server endpoint follows the pattern:
```
https://<account>.snowflakecomputing.com/api/v2/databases/<db>/schemas/<schema>/mcp-servers/<name>/sse
```

`langchain-mcp-adapters` supports SSE transport directly — no local bridge or npx needed.

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# MCP server endpoint — points to our MARKETING_MCP server object in Snowflake
# Snowflake hostnames require hyphens (not underscores) for valid SSL certs
MCP_HOST = f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com".replace('_', '-')
MCP_ENDPOINT = (
    f"https://{MCP_HOST}"
    f"/api/v2/databases/CORTEX_GATEWAY_LAB/schemas/PUBLIC/mcp-servers/MARKETING_MCP"
)

# Streamable HTTP transport — connects directly to the Snowflake MCP server
mcp_config = {
    "snowflake": {
        "transport": "http",
        "url": MCP_ENDPOINT,
        "headers": {"Authorization": f"Bearer {SNOWFLAKE_PAT}"},
    }
}

print("MCP server configured (will connect when agent starts)")
print(f"  Transport: http")
print(f"  Endpoint:  {MCP_ENDPOINT}")

MCP server configured (will connect when agent starts)
  Transport: http
  Endpoint:  https://SFSENORTHAMERICA-PERICKSON-AWS1.snowflakecomputing.com/api/v2/databases/CORTEX_GATEWAY_LAB/schemas/PUBLIC/mcp-servers/MARKETING_MCP


## 5. Build the LangChain Agent

We create a **ReAct agent** using LangGraph. The agent:
1. Receives a user question
2. Decides which MCP tool(s) to call (Cortex Analyst for data, Cortex Search for docs)
3. Routes LLM inference through the Cortex AI Gateway
4. Synthesizes a final answer from the tool results

In [5]:
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = """You are a marketing analytics assistant with access to campaign performance
data and strategy documents via Snowflake. Use the available tools to answer questions:

- For quantitative questions about campaign spend, revenue, ROI, etc.:
  1. First call query_campaigns to generate the SQL query.
  2. Extract the SQL statement from the response.
  3. Then call execute_sql with that SQL to get the actual data rows.
- For questions about strategy, methodology, or planning, use search_strategy_docs.
- For questions that need both data and context, use both.

Always cite specific numbers when available. Express ROI as a multiplier (e.g., 3.2x).
Round currency to 2 decimal places."""

# Initialize the MCP client and load tools once
mcp_client = MultiServerMCPClient(mcp_config)
tools = await mcp_client.get_tools()
print(f"Loaded {len(tools)} MCP tools: {[t.name for t in tools]}")

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)


async def run_agent(question: str):
    """Run the agent against a question."""
    result = await agent.ainvoke({"messages": [{"role": "user", "content": question}]})

    # Print tool calls and responses for visibility
    for msg in result["messages"]:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"Tool call: {tc['name']}({tc['args']})")
        if msg.type == 'tool':
            print(f"Tool response ({msg.name}): {msg.content[:500]}")
            print()

    final_message = result["messages"][-1]
    print("─" * 60)
    print(f"Q: {question}")
    print("─" * 60)
    print(final_message.content)
    return result

Loaded 3 MCP tools: ['query_campaigns', 'search_strategy_docs', 'execute_sql']


/var/folders/nx/tk8s9m3n36ncjzn4lvvk2dfw0000gn/T/ipykernel_35050/823485793.py:21: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


## 6. Run Queries

Let's ask the agent some marketing analytics questions. Each request flows through:

**User → LangChain Agent → AI Gateway (Claude) → MCP Tools (Snowflake) → Response**

All inference calls are logged by the gateway for observability.

In [6]:
# Query 1: Structured data question (routes to Cortex Analyst via MCP)
result1 = await run_agent("What was the ROAS by channel for Q4 2024?")

Tool call: query_campaigns({'message': 'What was the ROAS by channel for Q4 2024? Return channel, total spend, total revenue, and ROAS for campaigns in Q4 2024.'})
Tool response (query_campaigns): [{'type': 'text', 'text': '[{"text":"This is our interpretation of your question:\\n\\nWhat was the ROAS by channel for Q4 2024 (October, November, December 2024)? Return channel, total spend (rounded to 2 decimal places), total revenue (rounded to 2 decimal places), and ROAS (rounded to 2 decimal places) for campaigns in Q4 2024. Use official channel names: Paid Search, Social Media, Email, Display."},{"statement":"SELECT *\\nFROM SEMANTIC_VIEW(\\n    CORTEX_GATEWAY_LAB.PUBLIC.CMO_ANALYTICS\\n    DIMENSIONS channel\\n    METRICS total_spend, total_revenue, roas\\n    WHERE month >= \'2024-10-01\' AND month < \'2025-01-01\'\\n)\\n -- Generated by Cortex Analyst (request_id: 41295c82-cf26-41c5-a020-6dd448d811fc)\\n;","confidence":{}}]', 'id': 'lc_b0375314-9718-40e2-a200-294bd868d7a5'}]

Tool c

In [7]:
# Query 2: Unstructured data question (routes to Cortex Search via MCP)
result2 = await run_agent("What is our attribution methodology and how is revenue assigned to channels?")

Tool call: search_strategy_docs({'query': 'attribution methodology revenue assigned to channels attribution model channel revenue assignment methodology marketing', 'columns': ['content', 'title', 'attributes'], 'limit': 5})
Tool response (search_strategy_docs): [{'type': 'text', 'text': 'MCP error calling tool search_strategy_docs: Error occurred while calling Cortex Search Service: Column attributes was not indexed in this Cortex Search Service\nrequest-id: c49b4df6-1cdb-42f6-bcfd-766af7916627', 'id': 'lc_20810813-49ae-489b-83f9-0ec382ec5d4a'}]

Tool call: search_strategy_docs({'query': 'attribution methodology revenue assigned to channels attribution model channel revenue assignment methodology marketing', 'columns': ['content', 'title'], 'limit': 5})
Tool response (search_strategy_docs): [{'type': 'text', 'text': '[{"@scores":{"text_match":0.050141614,"cosine_similarity":0.56080717,"reranker_score":2.0358875},"title":"Attribution Methodology","content":"We use a data-driven multi-t

In [8]:
# Query 3: Hybrid question (may use both Analyst + Search)
result3 = await run_agent(
    "How did our Q4 Holiday Push campaign perform against the targets in the Q4 planning brief? "
    "Include actual ROAS and conversion numbers."
)

Tool call: query_campaigns({'message': 'How did our Q4 Holiday Push campaign perform? Return SQL to get actual spend, revenue, ROAS, and conversions for the Q4 Holiday Push campaign.'})
Tool call: search_strategy_docs({'query': 'Q4 planning brief Holiday Push targets ROAS conversions Q4 Holiday Push campaign', 'columns': ['content', 'title', 'attributes'], 'limit': 5})
Tool response (query_campaigns): [{'type': 'text', 'text': '[{"text":"This is our interpretation of your question:\\n\\nHow did the \'Q4 Holiday Push\' campaign perform in 2024? Return actual spend, revenue, ROAS, and conversions for the Q4 Holiday Push campaign, filtered to the full year 2024. Currency values rounded to 2 decimal places."},{"statement":"SELECT\\n    ROUND(total_spend, 2) AS total_spend,\\n    ROUND(total_revenue, 2) AS total_revenue,\\n    roas,\\n    total_conversions\\nFROM SEMANTIC_VIEW(\\n    CORTEX_GATEWAY_LAB.PUBLIC.CMO_ANALYTICS\\n    METRICS total_spend, total_revenue, roas, total_conversions\\n

## 7. Observability — AI Gateway Traces

Every request through the AI Gateway is recorded with OpenTelemetry traces. These land in
the `AGENT_TRACE_TABLE('SNOWFLAKE')` table function, where each row is a **span** in a trace.

Key columns:
- `TRACE_ID` — groups all spans from one end-to-end request
- `SPAN_ID` — unique identifier for each operation
- `RECORD_ATTRIBUTES` — JSON with model name, token counts, etc.
- `TIMESTAMP` — when the span was recorded

Because we enabled `capture_payload.request_response: true` in the gateway spec,
request/response content is also captured in the trace attributes.

Let's query the observability data for the requests we just made.

In [15]:
import snowflake.connector
import json
from pathlib import Path as _Path

# Fix corrupted CONFIG_MANAGER state from earlier failed connection attempt
from snowflake.connector.config_manager import CONFIG_MANAGER
for i, s in enumerate(CONFIG_MANAGER._slices):
    if isinstance(s.path, str):
        CONFIG_MANAGER._slices[i] = s._replace(path=_Path(s.path))

conn = snowflake.connector.connect(
    account=SNOWFLAKE_ACCOUNT,
    user=SNOWFLAKE_USER,
    authenticator="PROGRAMMATIC_ACCESS_TOKEN",
    token=SNOWFLAKE_PAT,
    warehouse="GATEWAY_LAB_WH",
)
cur = conn.cursor()
print(f"Connected as {SNOWFLAKE_USER} via PAT.")

Connected as PERICKSON via PAT.


In [16]:
# Recent AI Gateway traces
cur.execute("""
    SELECT
        TRACE:trace_id::STRING AS TRACE_ID,
        RECORD:name::STRING AS SPAN_NAME,
        RECORD_ATTRIBUTES:"gen_ai.request.model"::STRING AS MODEL,
        RECORD_ATTRIBUTES:"gen_ai.usage.input_tokens"::INT AS IN_TOK,
        RECORD_ATTRIBUTES:"gen_ai.usage.output_tokens"::INT AS OUT_TOK,
        RECORD_ATTRIBUTES:"http.status_code"::INT AS HTTP
    FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
    WHERE TIMESTAMP > DATEADD('hour', -1, CURRENT_TIMESTAMP())
    ORDER BY TIMESTAMP DESC
    LIMIT 15
""")

rows = cur.fetchall()
print(f"Found {len(rows)} recent gateway trace spans\n")
print(f"{'TRACE_ID':>36} | {'SPAN_NAME':>25} | {'MODEL':>16} | {'IN':>6} | {'OUT':>6} | {'HTTP':>4}")
print("─" * 100)
for r in rows:
    print(f"{str(r[0]):>36} | {str(r[1]):>25} | {str(r[2]):>16} | {r[3]:>6} | {r[4]:>6} | {r[5]:>4}")

Found 15 recent gateway trace spans

                            TRACE_ID |                 SPAN_NAME |            MODEL |     IN |    OUT | HTTP
────────────────────────────────────────────────────────────────────────────────────────────────────
    000000007903fe58e45fade4dc688abe |       chat openai-gpt-5.4 |   openai-gpt-5.4 |   2578 |    239 |  200
    000000001c0f2c9b8b22f5d612444cbc |       chat openai-gpt-5.4 |   openai-gpt-5.4 |   1617 |     43 |  200
    00000000cb43318d9df7ce52c22b67f0 |       chat openai-gpt-5.4 |   openai-gpt-5.4 |   1203 |    180 |  200
    000000008105002ef2c1b42a979aa0a5 |       chat openai-gpt-5.4 |   openai-gpt-5.4 |    783 |    107 |  200
    0000000097d591e6175a4f16259df67c |       chat openai-gpt-5.4 |   openai-gpt-5.4 |   1842 |    290 |  200
    00000000182bc989b4ccd9f9a746835a |       chat openai-gpt-5.4 |   openai-gpt-5.4 |    879 |     42 |  200
    00000000e0ae16e95f89da3ea8373a3c |       chat openai-gpt-5.4 |   openai-gpt-5.4 |    769 |     

In [17]:
# Credit usage summary from AI_GATEWAY_USAGE_HISTORY
# OPERATION_DETAILS is a JSON object keyed by model name with token counts
cur.execute("""
    SELECT
        f.key AS MODEL,
        COUNT(*) AS REQUEST_COUNT,
        SUM(f.value:"input_tokens"::INT) AS TOTAL_INPUT_TOKENS,
        SUM(f.value:"output_tokens"::INT) AS TOTAL_OUTPUT_TOKENS,
        SUM(CREDITS) AS TOTAL_CREDITS
    FROM SNOWFLAKE.ACCOUNT_USAGE.AI_GATEWAY_USAGE_HISTORY,
        LATERAL FLATTEN(input => OPERATION_DETAILS) f
    WHERE START_TIME > DATEADD('day', -1, CURRENT_TIMESTAMP())
    GROUP BY f.key
    ORDER BY TOTAL_CREDITS DESC
""")

rows = cur.fetchall()
print("AI Gateway Usage Summary (last 24h)\n")
print(f"{'MODEL':>20} | {'REQUESTS':>8} | {'IN_TOKENS':>10} | {'OUT_TOKENS':>10} | {'CREDITS':>10}")
print("─" * 70)
for row in rows:
    print(f"{str(row[0]):>20} | {row[1]:>8} | {row[2]:>10} | {row[3]:>10} | {row[4]:>10.6f}")

AI Gateway Usage Summary (last 24h)

               MODEL | REQUESTS |  IN_TOKENS | OUT_TOKENS |    CREDITS
──────────────────────────────────────────────────────────────────────
      openai-gpt-5.4 |       75 |      61509 |       7785 |   0.137130


In [18]:
# Detailed trace view — follow a single request end-to-end
cur.execute("""
    SELECT TRACE:trace_id::STRING AS TRACE_ID
    FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
    WHERE TIMESTAMP > DATEADD('hour', -1, CURRENT_TIMESTAMP())
    ORDER BY TIMESTAMP DESC
    LIMIT 1
""")
latest_trace = cur.fetchone()

if latest_trace:
    trace_id = latest_trace[0]
    print(f"Trace detail for: {trace_id}\n")

    cur.execute(f"""
        SELECT
            TRACE:span_id::STRING AS SPAN_ID,
            RECORD:name::STRING AS SPAN_NAME,
            TIMESTAMP,
            RECORD_ATTRIBUTES
        FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
        WHERE TRACE:trace_id::STRING = '{trace_id}'
        ORDER BY TIMESTAMP
    """)

    spans = cur.fetchall()
    for span in spans:
        attrs = json.loads(span[3]) if span[3] else {}
        print(f"  Span: {span[1]}")
        print(f"    Timestamp: {span[2]}")
        if "gen_ai.usage.input_tokens" in attrs:
            print(f"    Tokens: {attrs['gen_ai.usage.input_tokens']} in / {attrs['gen_ai.usage.output_tokens']} out")
        if "gen_ai.request.model" in attrs:
            print(f"    Model: {attrs['gen_ai.request.model']}")
        print()
else:
    print("No traces found yet — data may take a few minutes to appear.")

Trace detail for: 000000007903fe58e45fade4dc688abe

  Span: chat openai-gpt-5.4
    Timestamp: 2026-09-21 15:34:03.902915
    Tokens: 2578 in / 239 out
    Model: openai-gpt-5.4



In [19]:
# Deep dive: reconstruct the full agent conversation chain from a single trace
# This shows every reasoning step — tool calls, tool responses, and final answer.
cur.execute("""
    SELECT TRACE:trace_id::STRING AS TRACE_ID
    FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
    WHERE TIMESTAMP > DATEADD('hour', -2, CURRENT_TIMESTAMP())
      AND RECORD_ATTRIBUTES:"gen_ai.usage.input_tokens"::INT > 2000
    ORDER BY TIMESTAMP DESC
    LIMIT 1
""")
row = cur.fetchone()

if row:
    tid = row[0]
    cur.execute(f"""
        SELECT
            TRACE:span_id::STRING AS SPAN_ID,
            RECORD_ATTRIBUTES:"gen_ai.input.messages"::STRING AS INPUT_MSGS,
            RECORD_ATTRIBUTES:"gen_ai.output.messages"::STRING AS OUTPUT_MSGS,
            RECORD_ATTRIBUTES:"gen_ai.usage.input_tokens"::INT AS IN_TOK,
            RECORD_ATTRIBUTES:"gen_ai.usage.output_tokens"::INT AS OUT_TOK
        FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
        WHERE TRACE:trace_id::STRING = '{tid}'
        ORDER BY TIMESTAMP
    """)
    spans = cur.fetchall()
    print(f"Trace {tid} — {len(spans)} gateway call(s)\n")

    for i, span in enumerate(spans):
        msgs_in = json.loads(span[1]) if span[1] else []
        msgs_out = json.loads(span[2]) if span[2] else []
        print(f"═══ Gateway Call {i+1} ({span[3]} in / {span[4]} out tokens) ═══")

        for m in msgs_in:
            role = m.get('role', '?')
            parts = m.get('parts', [])
            for p in parts:
                if 'content' in p:
                    print(f"  [{role}] {p['content'][:200]}")
                elif 'name' in p:
                    print(f"  [{role}] → tool_call: {p['name']}({json.dumps(p.get('arguments',{}))[:150]})")
                elif 'response' in p:
                    resp_text = str(p['response'])[:200]
                    print(f"  [tool] ← {resp_text}")

        for m in msgs_out:
            parts = m.get('parts', [])
            for p in parts:
                if 'content' in p:
                    print(f"  [assistant] {p['content'][:300]}")
                elif 'name' in p:
                    print(f"  [assistant] → tool_call: {p['name']}({json.dumps(p.get('arguments',{}))[:150]})")
        print()
else:
    print("No multi-turn traces found in last 2 hours.")

Trace 000000007903fe58e45fade4dc688abe — 1 gateway call(s)

═══ Gateway Call 1 (2578 in / 239 out tokens) ═══
  [user] How did our Q4 Holiday Push campaign perform against the targets in the Q4 planning brief? Include actual ROAS and conversion numbers.
  [assistant] → tool_call: query_campaigns({"message": "How did our Q4 Holiday Push campaign perform? Return SQL to get actual spend, revenue, ROAS, and conversions for the Q4 Holiday Push camp)
  [assistant] → tool_call: search_strategy_docs({"columns": ["content", "title", "attributes"], "limit": 5, "query": "Q4 planning brief Holiday Push targets ROAS conversions Q4 Holiday Push campaign)
  [tool] ← [{'text': '[{"text":"This is our interpretation of your question:\\n\\nHow did the \'Q4 Holiday Push\' campaign perform in 2024? Return actual spend, revenue, ROAS, and conversions for the Q4 Holiday 
  [tool] ← [{'text': 'MCP error calling tool search_strategy_docs: Error occurred while calling Cortex Search Service: Column attributes wa

## Summary

This example demonstrated:

| Component | Role |
|---|---|
| **Cortex AI Gateway** | Centralized LLM routing — one endpoint, configurable routes, built-in observability |
| **LangChain + LangGraph** | Agent framework with ReAct reasoning over MCP tools |
| **MCP (Model Context Protocol)** | Standardized tool interface connecting the agent to Snowflake data services |
| **Cortex Analyst** | Natural language → SQL against the semantic view |
| **Cortex Search** | RAG retrieval over strategy documents |
| **AGENT_TRACE_TABLE** | OpenTelemetry traces for every gateway inference call |
| **AI_GATEWAY_USAGE_HISTORY** | Token usage and credit consumption per model/route |

### Key Takeaways

1. **The gateway is OpenAI-compatible** — any SDK or framework that speaks the OpenAI chat completions API works by pointing `base_url` at `/api/v2/cortex/v1`.
2. **Model allowlists control access** — the `models` spec restricts which models can be called; use `'*'` for unrestricted or `'claude-*'` to lock down to specific families.
3. **MCP provides a clean tool boundary** — the agent doesn't need Snowflake credentials or SQL knowledge; MCP handles that.
4. **Observability is automatic** — every inference call is traced in `AGENT_TRACE_TABLE('SNOWFLAKE')` with token counts, model metadata, and optional payload capture.